# 01. Dataset Analysis & Target Formulation
## Academic Project: Autonomous Warehouse AI — Predictive Analytics Component

### Objective
This notebook performs a comprehensive exploratory data audit of the warehouse inventory dataset. We:
1. Load raw warehouse inventory data (`data/raw/logistics_dataset.csv`).
2. Audit schema, data types, missing values, duplicates, and numerical distributions.
3. Diagnose the methodological failure of previous naive stockout rules (92.7% class collapse, 0.5092 ROC-AUC).
4. Formulate an academically rigorous, leak-free stockout risk target:
   $$\text{Target} = \mathbb{I}(\text{stock\_level} < \text{forecasted\_demand\_next\_7d})$$
5. Eliminate data leakage by isolating the target from the input feature space.
6. Conduct exploratory bivariate and multivariate analyses with publication-quality visualizations.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
sys.path.append(os.path.abspath("../src"))
from data_loader import load_raw_data, audit_dataset, get_prepared_dataframe

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
print("Libraries loaded successfully.")

### 1. Ingestion & Schema Inspection

In [ ]:
df_raw = load_raw_data()
print(f"Dataset Shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
print(f"Missing Values: {df_raw.isnull().sum().sum()}")
print(f"Duplicate Rows: {df_raw.duplicated().sum()}")
df_raw.head()

### 2. Descriptive Statistics

In [ ]:
df_raw.describe().round(2).T

### 3. Target Formulation & Leakage Elimination
In earlier iterations, stockout was formulated as `((stockout_count_last_month > 0) | (stock_level < reorder_point))`.
This induced an artificial 92.7% majority class collapse: models trivially predicted majority class, yielding 92.7% accuracy but random-guess **0.5092 ROC-AUC**.

Our principled target answers the operational question: **Does current physical stock satisfy projected demand over the replenishment horizon?**
$$\text{stock\_risk} = \begin{cases} 1 & \text{if } \text{stock\_level} < \text{forecasted\_demand\_next\_7d} \\ 0 & \text{otherwise} \end{cases}$$

Strict leakage prevention: `forecasted_demand_next_7d` is utilized **only** to establish the ground-truth target label and is permanently removed from the predictive feature space.

In [ ]:
df_labeled = get_prepared_dataframe()
target_counts = df_labeled["stock_risk"].value_counts()
print("Target Class Distribution:")
print(f"  0 (LOW RISK):  {target_counts[0]} ({target_counts[0]/len(df_labeled)*100:.2f}%)")
print(f"  1 (HIGH RISK): {target_counts[1]} ({target_counts[1]/len(df_labeled)*100:.2f}%)")
print(f"Class Imbalance Ratio: {target_counts[0] / target_counts[1]:.2f} : 1")

### 4. Target Distribution Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(["Low Risk (0)", "High Risk (1)"], [target_counts[0], target_counts[1]], color=["#2a9d8f", "#e76f51"], edgecolor="black", width=0.5)
ax.set_title("Warehouse Stock Risk Class Distribution", fontsize=12, fontweight="bold")
ax.set_ylabel("Item Count", fontsize=10)
for b in bars:
    h = b.get_height()
    ax.text(b.get_x() + b.get_width()/2., h + 30, f"{h} ({h/len(df_labeled)*100:.1f}%)", ha="center", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.show()

### 5. Bivariate Relationships with Stock Risk

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(data=df_labeled, x="stock_risk", y="stock_level", palette=["#2a9d8f", "#e76f51"], ax=axes[0])
axes[0].set_title("Current Stock Level by Risk Class", fontweight="bold")
axes[0].set_xticklabels(["Low Risk", "High Risk"])

sns.boxplot(data=df_labeled, x="stock_risk", y="daily_demand", palette=["#2a9d8f", "#e76f51"], ax=axes[1])
axes[1].set_title("Daily Demand by Risk Class", fontweight="bold")
axes[1].set_xticklabels(["Low Risk", "High Risk"])

plt.tight_layout()
plt.show()

### 6. Correlation Analysis & Audit Export

In [ ]:
audit_res = audit_dataset(df_raw)
print("Audit Report Summary:")
print(f"Target Variable: {audit_res['target_variable']}")
print(f"Distribution: {audit_res['target_distribution']}")